# Monte Carlo aggregation

Inputs (`{b}` = `closed`, `open`):
- `output/mc_samples.csv.gz`: accepted Monte Carlo pathways (notebook 01)
- `output/central_pathways.csv`: design pathways (notebook 01)
- `output/{b}/geological_lifetime_caps.csv`: geological lifetime caps on storage resource (notebook 02)
- `input/ar6_world_ccs_2050.xlsx`, `input/ar6_regional_ccs_2050.json`: IPCC AR6 CCS rates in 2050

## Settings

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'input').is_dir() and (ROOT.parent / 'input').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'input').is_dir(), 'Run from the Global Study folder or its code folder.'
OUT = ROOT / 'output'
BOUNDARIES = ['closed', 'open']
SCENARIOS = ['minimum', 'growth10', 'policy', 'us1gt', 'reference', 'maximum']
MODELS = ['Logistic', 'Gompertz']
KEYS = ['scenario', 'country', 'model']
N_DRAWS = 10_000
START_YEAR = 2030
HORIZONS = (2050, 2100, 2180)
CAP_RTOL = 1e-12

STAGES = ['pre', 'post', 'pre_all_jurisdictions']
METRICS = {'rate_2050': 'rate_2050_Gt_yr',
           **{f'stock_{y}': f'cumulative_{y}_Gt' for y in HORIZONS},
           **{f'rate_{y}': f'rate_{y}_Gt_yr' for y in HORIZONS[1:]}}
QUANTILES = {'p05': .05, 'p25': .25, 'p50': .5, 'p75': .75, 'p95': .95, 'p99': .99}
GZIP = {'method': 'gzip', 'mtime': 0}

## Sampled pathways

In [ ]:
samples = pd.read_csv(OUT / 'mc_samples.csv.gz', float_precision='round_trip')
central = pd.read_csv(OUT / 'central_pathways.csv', float_precision='round_trip').set_index(KEYS)
assert not samples.duplicated(KEYS + ['draw_id']).any()

POOLS = {}
for key, g in samples.groupby(KEYS, sort=False):
    g = g.sort_values('draw_id', kind='stable')
    POOLS[key] = {col: g[col].to_numpy(float) for col in ['C_Gt', 'r_per_yr', *METRICS.values()]}
COUNTRIES = sorted(central.index.unique('country'))
assert len(COUNTRIES) == 10 and set(POOLS) == set(central.index)

In [ ]:
def pathway(model, C, r, S0, year):
    e = np.exp(-r * (year - START_YEAR))
    if model == 'Logistic':
        S = C / (1 + (C / S0 - 1) * e)
        return S, r * S * (1 - S / C)
    b = np.log(C / S0)
    S = C * np.exp(-b * e)
    return S, r * S * b * e

for key, p in POOLS.items():
    assert (p['rate_2050_Gt_yr'] > 0).all() and (p['C_Gt'] < central.C_scenario_Gt[key]).all(), key
    for year in HORIZONS:
        S, Q = pathway(key[2], p['C_Gt'], p['r_per_yr'], central.S0_Gt[key], year)
        np.testing.assert_allclose(S, p[f'cumulative_{year}_Gt'], rtol=1e-12, atol=1e-12)
        np.testing.assert_allclose(Q, p[f'rate_{year}_Gt_yr'], rtol=1e-9, atol=1e-12)

## Coverage and seeds

In [ ]:
def load_caps(boundary):
    caps = pd.read_csv(OUT / boundary / 'geological_lifetime_caps.csv',
                       float_precision='round_trip').set_index(KEYS).C_cap_gt
    assert caps.index.is_unique and set(caps.index) == set(POOLS)
    assert (caps.isna() | (np.isfinite(caps) & (caps >= 0))).all()
    return caps

def included_jurisdictions(caps, scenario, model):
    included = [c for c in COUNTRIES if np.isfinite(caps[scenario, c, model])]
    assert included, (scenario, model)
    return included

def coverage(included):
    excluded = [c for c in COUNTRIES if c not in included]
    return dict(n_jurisdictions=len(included), included_jurisdictions='; '.join(included),
                excluded_jurisdictions='; '.join(excluded), coverage_complete=not excluded,
                aggregation_scope='available_jurisdictions' if excluded else 'complete_ten_jurisdictions')

def unavailable(included):
    return ';'.join(c for c in COUNTRIES if c not in included)

def seed(*parts):
    return int.from_bytes(hashlib.sha256('|'.join(map(str, parts)).encode()).digest()[:4], 'little')

## Aggregation

In [ ]:
def aggregate(caps):
    groups = {}
    for scenario in SCENARIOS:
        for model in MODELS:
            included = included_jurisdictions(caps, scenario, model)
            total = {(stage, metric): np.zeros(N_DRAWS) for stage in STAGES for metric in METRICS}
            for country in COUNTRIES:
                key = (scenario, country, model)
                pool = POOLS[key]
                idx = np.random.default_rng(seed(*key)).integers(0, len(pool['C_Gt']), N_DRAWS)
                picks = {'pre_all_jurisdictions': idx}
                if country in included:
                    keep = pool['C_Gt'] <= caps[key] * (1 + CAP_RTOL)
                    if keep.all():
                        idx_post = idx
                    else:
                        kept = np.flatnonzero(keep)
                        assert len(kept), f'no draw within the cap: {key}'
                        idx_post = kept[np.random.default_rng(seed(*key, 'feasible', 'min'))
                                        .integers(0, len(kept), N_DRAWS)]
                    picks.update(pre=idx, post=idx_post)
                for metric, column in METRICS.items():
                    for stage, ix in picks.items():
                        total[stage, metric] += pool[column][ix]
            groups[scenario, model] = included, total
    return groups

## Tables

In [ ]:
DIST_METRICS = ['rate_2050'] + [f'stock_{y}' for y in HORIZONS]
DIST_COLUMNS = [f'{stage}_{m}' for m in DIST_METRICS for stage in STAGES]
RATE_COLUMNS = [f'{stage}_rate_{y}' for stage in STAGES for y in HORIZONS]

def draw_tables(groups):
    frames = []
    for (scenario, model), (included, total) in groups.items():
        frame = pd.DataFrame(dict(scenario=scenario, model=model, draw_id=np.arange(N_DRAWS),
                                  post_aggregate_available=True, unavailable_countries=unavailable(included),
                                  **coverage(included)))
        frame = frame.join(pd.DataFrame({f'{stage}_{m}': v for (stage, m), v in total.items()}))
        frames.append(frame)
    draws = pd.concat(frames, ignore_index=True)
    meta = [c for c in draws.columns if c not in DIST_COLUMNS + RATE_COLUMNS]
    return draws[meta + DIST_COLUMNS], draws[meta + RATE_COLUMNS]

def distribution_summary(groups):
    rows = []
    for (scenario, model), (included, total) in groups.items():
        for metric in DIST_METRICS:
            q = {stage: np.quantile(total[stage, metric], [.05, .5, .95]) for stage in STAGES}
            rows.append(dict(
                scenario=scenario, model=model, metric=metric,
                units='Gt/yr' if metric.startswith('rate') else 'Gt',
                n_countries_pre=len(included), n_countries_post=len(included),
                n_countries_pre_all_jurisdictions=len(COUNTRIES),
                post_aggregate_available=True, unavailable_countries=unavailable(included),
                **{f'{stage}_{p}': v for stage in STAGES for p, v in zip(['p05', 'p50', 'p95'], q[stage])},
                reduction_pct=100 * (q['pre'][1] - q['post'][1]) / q['pre'][1] if q['pre'][1] else 0.,
                arrays_identical=bool(np.array_equal(total['pre', metric], total['post', metric])),
                **coverage(included)))
    return pd.DataFrame(rows)

def rate_horizon_summary(groups):
    rows = []
    for (scenario, model), (included, total) in groups.items():
        for stage in STAGES:
            full = stage == 'pre_all_jurisdictions'
            for year in HORIZONS:
                v = total[stage, f'rate_{year}']
                rows.append(dict(
                    scenario=scenario, model=model, stage=stage, year=year,
                    n=len(v), n_available=int(np.isfinite(v).sum()), post_aggregate_available=True,
                    unavailable_countries='' if full else unavailable(included),
                    minimum=float(np.amin(v)), maximum=float(np.amax(v)),
                    **coverage(COUNTRIES if full else included),
                    **{name: float(np.quantile(v, q)) for name, q in QUANTILES.items()}))
    return pd.DataFrame(rows)

def jurisdiction_rate_summary(caps):
    rows = []
    for scenario in SCENARIOS:
        for country in COUNTRIES:
            for model in MODELS:
                key = (scenario, country, model)
                rates = POOLS[key]['rate_2050_Gt_yr']
                pools = {'pre': rates}
                if np.isfinite(caps[key]):
                    pools['post'] = rates[POOLS[key]['C_Gt'] <= caps[key] * (1 + CAP_RTOL)]
                for stage in ['pre', 'post']:
                    v = pools.get(stage)
                    q = np.full(5, np.nan) if v is None else np.quantile(v, [.05, .25, .5, .75, .95])
                    rows.append(dict(
                        scenario=scenario, country=country, model=model, stage=stage,
                        available=v is not None, n=0 if v is None else len(v),
                        fixed_target=central.target_type[key] == 'fixed',
                        **dict(zip(['p05', 'p25', 'p50', 'p75', 'p95'], q)),
                        minimum=np.nan if v is None else float(np.amin(v)),
                        maximum=np.nan if v is None else float(np.amax(v))))
    return pd.DataFrame(rows)

## Run both boundaries

In [ ]:
for boundary in BOUNDARIES:
    folder = OUT / boundary
    caps = load_caps(boundary)
    groups = aggregate(caps)
    dist_draws, rate_draws = draw_tables(groups)
    dist_draws.to_csv(folder / 'global_distribution_draws.csv.gz', index=False, compression=GZIP)
    rate_draws.to_csv(folder / 'global_rate_horizon_draws.csv.gz', index=False, compression=GZIP)
    summary = distribution_summary(groups)
    summary.to_csv(folder / 'global_distribution_summary.csv', index=False)
    rate_horizon_summary(groups).to_csv(folder / 'global_rate_horizon_summary.csv', index=False)
    jurisdiction_rate_summary(caps).to_csv(folder / 'jurisdiction_rate2050_summary.csv', index=False)
    covered = pd.DataFrame([dict(boundary=boundary, scenario=s, model=m, **coverage(inc))
                            for (s, m), (inc, _) in groups.items()])
    covered.to_csv(folder / 'aggregation_coverage.csv', index=False)

    print(f'{boundary}: partial coverage')
    display(covered.loc[~covered.coverage_complete, ['scenario', 'model', 'n_jurisdictions', 'excluded_jurisdictions']])
    display(summary.query("metric == 'rate_2050'")[['scenario', 'model', 'pre_p50', 'post_p50', 'reduction_pct']].round(3))

## AR6 2050 storage rates

In [ ]:
world = pd.read_excel(ROOT / 'input/ar6_world_ccs_2050.xlsx', sheet_name='World_2050')
assert world['Region/Country'].eq('World').all()
world = pd.DataFrame({'model': world['Model'], 'scenario': world['Scenario'],
                      'category': world['Climate Pathways in AR6 Database'].str.extract(r'^(C[1-4]):', expand=False),
                      'rate_gt_yr': world['2050 Storage Rate [Gt/yr]']})
assert world.category.notna().all() and (np.isfinite(world.rate_gt_yr) & (world.rate_gt_yr >= 0)).all()
world.to_csv(OUT / 'ar6_world_rate2050.csv', index=False)

regional = pd.DataFrame(json.loads((ROOT / 'input/ar6_regional_ccs_2050.json').read_text())['rows'])
regional = regional.rename(columns={'ar6_category': 'category'})[
    ['study_jurisdiction', 'source_region', 'model', 'scenario', 'category', 'rate_gt_yr']]
assert regional.category.isin(['C1', 'C2', 'C3', 'C4']).all()
assert (np.isfinite(regional.rate_gt_yr) & (regional.rate_gt_yr >= 0)).all()
regional.to_csv(OUT / 'ar6_regional_rate2050.csv', index=False)

display(world.groupby('category').rate_gt_yr.describe().round(3))
display(regional.groupby('study_jurisdiction').rate_gt_yr.describe().round(3))